# Tech Challenge - Fase 3: Deploy de Modelo em Produção
## Triagem Automática de Laudos Médicos com NLP (EDA e Modelagem Otimizada)

**Tema Central:** Deploy de Modelo em Produção com Pipeline CI/CD, Monitoramento e Otimização de Latência.  
**Objetivo deste Notebook:**
1. Realizar a Análise Exploratória de Dados (EDA) dos laudos médicos (`medical_abstract`) e das 5 classes clínicas (`condition_label`).
2. Investigar a causa-raiz da sobreposição entre classes (especialmente a classe genérica *General Pathological Conditions* vs. doenças específicas).
3. Implementar técnicas de engenharia de atributos (TF-IDF com sublinear scaling, n-gramas e remoção de boilerplate clínico).
4. Avaliar estratégias de tratamento de desbalanceamento (`class_weight='balanced'`) para priorizar o **Recall clínico** das condições graves (cardiovascular, neoplasias, sistema nervoso e digestivo).
5. Comparar múltiplos classificadores lineares leves (Scikit-Learn).
6. Medir o baseline de latência de inferência local (P50, P95, P99).
7. Serializar o pipeline campeão em `models/baseline_model.joblib` para servir de motor da **API FastAPI (Etapa 1)**.


## 1. Importação de Bibliotecas e Setup


In [ ]:
import os
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Ambiente configurado com sucesso!")


## 2. Carregamento dos Dados

Ingestão dos três arquivos presentes no diretório `raw_data`:
* `medical_tc_labels.csv`: Mapeamento das 5 classes diagnósticas.
* `medical_tc_train.csv`: Conjunto de treino (11.550 amostras).
* `medical_tc_test.csv`: Conjunto de teste (2.888 amostras).


In [ ]:
DATA_DIR = os.path.join("..", "raw_data") if os.path.exists(os.path.join("..", "raw_data")) else "raw_data"
LABELS_PATH = os.path.join(DATA_DIR, "medical_tc_labels.csv")
TRAIN_PATH = os.path.join(DATA_DIR, "medical_tc_train.csv")
TEST_PATH = os.path.join(DATA_DIR, "medical_tc_test.csv")

df_labels = pd.read_csv(LABELS_PATH)
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

label_map = dict(zip(df_labels['condition_label'], df_labels['condition_name']))
df_train['condition_name'] = df_train['condition_label'].map(label_map)
df_test['condition_name'] = df_test['condition_label'].map(label_map)

print(f"Treino: {df_train.shape[0]} amostras | Teste: {df_test.shape[0]} amostras")
display(df_labels)


## 3. Qualidade dos Dados e Integridade


In [ ]:
print("--- Valores Ausentes ---")
print("Treino:", df_train.isnull().sum().to_dict())
print("Teste: ", df_test.isnull().sum().to_dict())

print("\n--- Textos Duplicados ---")
print(f"Treino: {df_train['medical_abstract'].duplicated().sum()} duplicatas")
print(f"Teste:  {df_test['medical_abstract'].duplicated().sum()} duplicatas")


## 4. Distribuição das Classes e Análise do Desbalanceamento

A classe 5 (*general pathological conditions*) representa mais de 33% do conjunto de dados, enquanto a classe 2 (*digestive system diseases*) representa apenas ~10%. Esse desbalanceamento de 3.3x afeta diretamente classificadores não balanceados, fazendo-os favorecer a classe majoritária.


In [ ]:
dist_train = df_train['condition_name'].value_counts()
dist_test = df_test['condition_name'].value_counts()

df_dist = pd.DataFrame({
    'Treino (Qtd)': dist_train,
    'Treino (%)': (dist_train / len(df_train) * 100).round(2),
    'Teste (Qtd)': dist_test,
    'Teste (%)': (dist_test / len(df_test) * 100).round(2),
})

display(df_dist)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

sns.barplot(x=dist_train.values, y=dist_train.index, ax=axes[0], palette='Blues_r', hue=dist_train.index, legend=False)
axes[0].set_title('Distribuição no Treino (N=11.550)')
axes[0].set_xlabel('Contagem')
for i, v in enumerate(dist_train.values):
    axes[0].text(v + 30, i, f"{v} ({v/len(df_train)*100:.1f}%)", va='center')

sns.barplot(x=dist_test.values, y=dist_test.index, ax=axes[1], palette='Oranges_r', hue=dist_test.index, legend=False)
axes[1].set_title('Distribuição no Teste (N=2.888)')
axes[1].set_xlabel('Contagem')
for i, v in enumerate(dist_test.values):
    axes[1].text(v + 10, i, f"{v} ({v/len(df_test)*100:.1f}%)", va='center')

plt.tight_layout()
plt.show()


## 5. Análise de Comprimento e Vocabulário dos Laudos Médicos


In [ ]:
for df in [df_train, df_test]:
    df['char_length'] = df['medical_abstract'].astype(str).str.len()
    df['word_count'] = df['medical_abstract'].astype(str).str.split().str.len()

display(df_train[['char_length', 'word_count']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_train['word_count'], bins=50, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Distribuição de Palavras por Laudo')
axes[0].set_xlabel('Número de Palavras')
axes[0].axvline(df_train['word_count'].median(), color='red', linestyle='--', label=f"Mediana: {df_train['word_count'].median():.0f}")
axes[0].legend()

sns.boxplot(data=df_train, x='word_count', y='condition_name', ax=axes[1], palette='Set2')
axes[1].set_title('Comprimento por Especialidade Clínica')
axes[1].set_xlabel('Número de Palavras')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


## 6. Engenharia de Features e Limpeza de Ruído Clínico

Laudos médicos possuem termos recorrentes neutros (*"patient", "study", "cases", "methods", "results"*) que agregam pouco valor discriminativo e consomem peso no TF-IDF. Além disso, termos médicos essenciais são compostos (*"coronary artery", "myocardial infarction"*).

Definimos:
* **Stopwords Clínicas:** Remoção de stopwords em inglês + jargão estrutural de laudos.
* **N-gramas:** Unigramas e bigramas `(1, 2)`.
* **Filtros de Frequência:** `min_df=2` (elimina erros tipográficos raros) e `max_df=0.75` (elimina termos onipresentes).
* **Escalação Sublinear:** `sublinear_tf=True` (suaviza a influência de termos repetidos no mesmo laudo via $1 + \log(TF)$).


In [ ]:
clinical_stopwords = list(ENGLISH_STOP_WORDS.union({
    'patient', 'patients', 'case', 'cases', 'study', 'studies', 'results',
    'result', 'conclusion', 'conclusions', 'method', 'methods', 'treated',
    'treatment', 'underwent', 'using', 'used', 'clinical', 'associated',
    'group', 'data', 'years', 'age', 'year', 'performed', 'evaluated'
}))

print(f"Total de stopwords combinadas: {len(clinical_stopwords)}")


## 7. Comparativo de Modelos: Efeito do Balanceamento e Algoritmos

Comparamos diferentes abordagens lineares sob duas óticas fundamentais:
1. **Acurácia Global vs. F1-Score Macro:** Em triagem médica, o **F1-Macro** é mais representativo, pois dá igual importância ao acerto em todas as especialidades.
2. **Impacto do `class_weight='balanced'`:** Força o estimador a penalizar severamente falsos negativos nas especialidades minoritárias.


In [ ]:
vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.75,
    stop_words=clinical_stopwords,
    ngram_range=(1, 2),
    sublinear_tf=True
)

candidate_models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
    "Complement Naive Bayes": ComplementNB(alpha=0.5),
    "Regressão Logística (Padrão)": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Regressão Logística (Balanced, C=2.0)": LogisticRegression(C=2.0, class_weight='balanced', max_iter=1000, random_state=42),
    "SGD Classifier (Log Loss Balanced)": SGDClassifier(loss='log_loss', class_weight='balanced', alpha=1e-4, max_iter=1000, random_state=42),
    "Ridge Classifier": RidgeClassifier(alpha=1.0, random_state=42)
}

results = []
trained_pipelines = {}

X_train, y_train = df_train['medical_abstract'], df_train['condition_label']
X_test, y_test = df_test['medical_abstract'], df_test['condition_label']

for name, clf in candidate_models.items():
    pipe = Pipeline([
        ('vec', vectorizer),
        ('clf', clf)
    ])
    
    t0_tr = time.time()
    pipe.fit(X_train, y_train)
    t_tr = time.time() - t0_tr
    
    t0_pred = time.time()
    preds = pipe.predict(X_test)
    t_pred = time.time() - t0_pred
    
    acc = accuracy_score(y_test, preds)
    f1_m = f1_score(y_test, preds, average='macro')
    f1_w = f1_score(y_test, preds, average='weighted')
    lat_ms = (t_pred / len(X_test)) * 1000
    
    trained_pipelines[name] = pipe
    results.append({
        'Modelo': name,
        'Acurácia': round(acc, 4),
        'F1-Score Macro': round(f1_m, 4),
        'F1-Score Weighted': round(f1_w, 4),
        'Tempo Treino (s)': round(t_tr, 2),
        'Latência (ms/doc)': round(lat_ms, 3)
    })

df_results = pd.DataFrame(results).sort_values(by='F1-Score Macro', ascending=False)
display(df_results)


## 8. Avaliação Detalhada do Modelo Campeão e Análise de Erros

Selecionamos a **Regressão Logística com `class_weight='balanced'` e `C=2.0`**.  
Embora a acurácia global fique em torno de **57.5%**, o ganho em **Recall clínico** para doenças específicas é expressivo:
* **Neoplasias:** Recall de **~73%** (antes ~66%).
* **Cardiovascular:** Recall de **~71%** (antes ~61%).
* **Sistema Nervoso:** Recall de **~60%** (antes ~41%).
* **Sistema Digestivo:** Recall de **~60%** (antes ~40%).


In [ ]:
champion_name = "Regressão Logística (Balanced, C=2.0)"
champion_pipe = trained_pipelines[champion_name]

y_pred_champion = champion_pipe.predict(X_test)
target_names = [label_map[i] for i in sorted(label_map.keys())]

print(f"=== Classification Report - {champion_name} ===")
print(classification_report(y_test, y_pred_champion, target_names=target_names, digits=4))


In [ ]:
cm = confusion_matrix(y_test, y_pred_champion, normalize='true')

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', ax=ax, values_format='.2f', xticks_rotation=45)
ax.set_title(f'Matriz de Confusão Normalizada - {champion_name}')
plt.grid(False)
plt.tight_layout()
plt.show()


## 9. Benchmark de Latência Unitária Local

Simulação de chamadas individuais (como ocorrerá via endpoint REST da FastAPI) com aquecimento (*warm-up*) e medição percentilar.


In [ ]:
sample_text = X_test.iloc[0]
print(f"Laudo de amostra: '{sample_text[:110]}...'")

# Warm-up
for _ in range(15):
    _ = champion_pipe.predict([sample_text])

# Benchmark de 250 inferências
latencies = []
for _ in range(250):
    t_start = time.perf_counter()
    _ = champion_pipe.predict([sample_text])
    latencies.append((time.perf_counter() - t_start) * 1000)

latencies = np.array(latencies)
print(f"--- Latência Unitária (CPU Local) ---")
print(f"Média:         {latencies.mean():.3f} ms")
print(f"Mediana (P50): {np.percentile(latencies, 50):.3f} ms")
print(f"P95:           {np.percentile(latencies, 95):.3f} ms")
print(f"P99:           {np.percentile(latencies, 99):.3f} ms")


## 10. Exportação do Pipeline e Metadados para a API (Etapa 1)

Exportação de:
1. `models/baseline_model.joblib`: Pipeline treinado com vetorizador e classificador balanceado.
2. `models/classes_mapping.json`: Mapeamento de rótulos, metadados e métricas para consumo da API.
3. *Sanity Check* com inferência real e probabilidades calibradas.


In [ ]:
MODELS_DIR = os.path.join("..", "models") if os.path.exists(os.path.join("..", "models")) else "models"
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODELS_DIR, "baseline_model.joblib")
METADATA_PATH = os.path.join(MODELS_DIR, "classes_mapping.json")

# 1. Salvar pipeline
joblib.dump(champion_pipe, MODEL_PATH)
print(f"Modelo exportado: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.1f} KB)")

# 2. Salvar metadados
metadata = {
    "model_name": "Tfidf_LogisticRegression_Balanced",
    "version": "1.1.0",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "labels_map": {int(k): v for k, v in label_map.items()},
    "metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred_champion)),
        "f1_macro": float(f1_score(y_test, y_pred_champion, average='macro')),
        "f1_weighted": float(f1_score(y_test, y_pred_champion, average='weighted')),
        "p95_latency_ms": float(np.percentile(latencies, 95))
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print(f"Metadados exportados: {METADATA_PATH}")

# 3. Sanity Check de inferência
loaded_pipe = joblib.load(MODEL_PATH)
test_sample = ["Patient admitted with severe chest pain, elevated troponin levels and acute myocardial infarction."]
pred_id = int(loaded_pipe.predict(test_sample)[0])
pred_probs = loaded_pipe.predict_proba(test_sample)[0]
confidence = float(np.max(pred_probs))
condition_desc = metadata['labels_map'][pred_id]

print(f"\n--- Sanity Check ---")
print(f"Entrada: {test_sample[0]}")
print(f"Predição: Classe {pred_id} ({condition_desc}) - Confiança: {confidence:.2%}")


## 11. Conclusões e Roadmap da Etapa 1

### Conclusões da Análise:
* **Diagnóstico de Performance:** Em datasets médicos multiclasse com rótulos genéricos (como OHSUMED), a acurácia nominal (~58%) reflete o alto grau de sobreposição semântica entre doenças e a classe geral.
* **Ganho Clínico:** A introdução de `class_weight='balanced'` e limpeza de jargão aumentou a sensibilidade (Recall) para doenças cardiovasculares, neoplasias, digestivas e neurológicas para **60% - 73%**, aspecto primordial para triagem de urgência hospitalar.
* **Adequação ao Requisito de Latência:** O modelo atinge tempo de inferência unitário de **~0.3 ms a 0.5 ms**, ideal para atender às exigências de SLAs rigorosos em produção.

### Próximos Passos (Etapa 1 - Decisão Arquitetural e API Inicial):
1. **API FastAPI (`src/api/`):** Construir a aplicação REST com endpoints `/predict` e `/health`.
2. **Conteinerização (`Dockerfile`):** Empacotar a API e dependências em contêiner Docker leve.
3. **Medição de Tempo de Resposta Local:** Medir a latência HTTP do contêiner e definir o baseline da aplicação.
4. **Documentação no [`README.md`](file:///c:/Users/a912453/OneDrive%20-%20ATOS/Documentos/fiap_mle/fase3/README.md):** Registrar a análise e decisão da estratégia em nuvem (Batch vs. Real-time).
